# Symmetry Detection

In [ ]:
import sys
import time
from pathlib import Path
import open3d as o3d
import trimesh
import numpy as np
sys.path.insert(0, '../')
import drm
from drm.detect import detect_symmetry_o3d, symmetry_geometries, reflect_pointcloud

%load_ext autoreload
%autoreload 2

path = "/home/jvermandere/datasets/V-Scan/data/Office_1_Leica-P30_1775812376935/Chair Guest Green(Clone)1775812376937_points.txt"

# Works directly with your txt_pcd_to_open3d() output
pcd, matrix = drm.txt_pcd_to_open3d(path)
drm.visualise_open3d(pcd).show()

In [ ]:

result = detect_symmetry_o3d(pcd, threshold=0.02, n_random_candidates=600)
print(result)
# → plane_normal, plane_point, score, threshold


In [ ]:

# Get Open3D geometries for your visualiser
geoms = symmetry_geometries(result, pcd)          # axis + plane + reflected cloud
scene = drm.visualise_open3d([pcd] + geoms)       # your existing helper
scene.show()


In [ ]:
# Or get the mirrored cloud as a proper o3d PointCloud (colors + normals preserved)
pcd_reflected = reflect_pointcloud(pcd, result)
drm.visualise_open3d(pcd_reflected).show()

## Shapenet experiments

In [ ]:
import os
import glob
import time
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple
 
import numpy as np
import open3d as o3d
 
from drm.detect import (
    detect_multi_symmetry_o3d,
    MultiSymmetryResult,
    SymmetryResult,
    _plane_normal_angle_deg,
)
 
 
# ── Known ShapeNet synset → human-readable name ────────────────────────────────
SHAPENET_SYNSETS: Dict[str, str] = {
    "02691156": "airplane",
    "02828884": "bench",
    "02933112": "cabinet",
    "02958343": "car",
    "03001627": "chair",
    "03211117": "display",
    "03636649": "lamp",
    "03691459": "loudspeaker",
    "03790512": "motorbike",
    "03948459": "pistol",
    "04090263": "rifle",
    "04256520": "sofa",
    "04379243": "table",
    "04401088": "telephone",
    "04530566": "watercraft",
}
 
 
# ═══════════════════════════════════════════════════════════════════════════════
#  Result dataclasses
# ═══════════════════════════════════════════════════════════════════════════════
 
@dataclass
class AxisMatchResult:
    """
    One partial-cloud axis matched against its nearest full-cloud axis.
 
    The partial axis is the detected one; the full axis is the reference it
    was matched to (lowest angular distance among all full-cloud axes).
    Both axes are ranked by detection score within their own cloud.
    """
    partial_rank:   int          # 0 = best partial axis, 1 = second best, …
    partial_normal: np.ndarray   # (3,) unit normal from partial cloud
    partial_score:  float        # symmetry score on partial cloud
 
    full_rank:      int          # which full-cloud axis this matched (0 = best)
    full_normal:    np.ndarray   # (3,) unit normal from full cloud (closest match)
    full_score:     float        # symmetry score on full cloud
 
    angular_error_deg: float     # angle between partial and matched full normal [0–90°]
    found:          bool         # True if angular_error_deg < max_match_deg
 
 
@dataclass
class ShapeSymmetryResult:
    """Symmetry comparison result for a single shape (multi-axis, NN matching)."""
    obj_path:           str
 
    multi_full:         MultiSymmetryResult   # axes on full cloud (reference)
    n_points_full:      int
 
    multi_partial:      MultiSymmetryResult   # axes on partial cloud
    n_points_partial:   int
    camera_pos:         np.ndarray
 
    # One entry per partial axis (len = multi_partial.k), ordered by partial score
    axis_matches:       List[AxisMatchResult]
 
    # Convenience aggregates over found matches only
    errors_deg:         List[float]   # angular_error_deg for each partial axis (found or 90°)
    found_fraction:     float         # fraction of partial axes within max_match_deg
    mean_score_full:    float
    mean_score_partial: float
 
    elapsed_full_s:     float
    elapsed_partial_s:  float
 
    def __str__(self) -> str:
        name = Path(self.obj_path).parents[1].name
        lines = [
            f"ShapeSymmetryResult [{name}]",
            f"  pts full/partial    : {self.n_points_full} / {self.n_points_partial}  "
            f"({self.n_points_partial/self.n_points_full*100:.0f}% visible)",
            f"  axes full / partial : {self.multi_full.k} / {self.multi_partial.k}",
            f"  {'#':>2}  {'partial normal':>22}  sc_p  →  {'full normal':>22}  sc_f  err°  ok",
        ]
        for m in self.axis_matches:
            tag = "✓" if m.found else "✗"
            pn  = f"[{m.partial_normal[0]:+.2f},{m.partial_normal[1]:+.2f},{m.partial_normal[2]:+.2f}]"
            fn  = f"[{m.full_normal[0]:+.2f},{m.full_normal[1]:+.2f},{m.full_normal[2]:+.2f}]"
            lines.append(
                f"  {m.partial_rank:>2}  {pn:>22}  {m.partial_score:.2f}  →  "
                f"{fn:>22}  {m.full_score:.2f}  {m.angular_error_deg:5.1f}°  {tag}"
            )
        lines.append(
            f"  found {self.found_fraction*100:.0f}% of partial axes  |  "
            f"errors: {[f'{e:.1f}°' for e in self.errors_deg]}"
        )
        return "\n".join(lines)
 
 
@dataclass
class ClassBenchmarkResult:
    """Aggregated symmetry benchmark for one ShapeNet class."""
    synset_id:      str
    class_name:     str
    n_shapes:       int
    k:              int            # max partial axes compared per shape
 
    # Per partial-axis-rank errors (index 0 = primary axis, 1 = secondary, …)
    # Each list has length = number of shapes that had that axis
    errors_by_rank: List[List[float]]   # errors_by_rank[rank][shape_idx]
 
    # Aggregate over all partial axes across all shapes
    mean_angular_error_deg:  float
    std_angular_error_deg:   float
 
    # Per-rank summaries
    mean_error_by_rank:  List[float]   # mean error for rank-0, rank-1, …
    found_rate_by_rank:  List[float]   # fraction found (< max_match_deg) per rank
 
    mean_fraction_visible:   float
    mean_score_full:         float
    mean_score_partial:      float
 
    per_shape: List[ShapeSymmetryResult] = field(default_factory=list, repr=False)
 
    def __str__(self) -> str:
        lines = [f"═══ {self.class_name} ({self.synset_id}) | n={self.n_shapes} | k={self.k} ═══"]
        lines.append(f"  Overall mean error : {self.mean_angular_error_deg:.2f}° ± {self.std_angular_error_deg:.2f}°")
        for rank, (mean_e, found_r) in enumerate(zip(self.mean_error_by_rank, self.found_rate_by_rank)):
            label = {0: "primary", 1: "secondary", 2: "tertiary"}.get(rank, f"rank-{rank}")
            n = len(self.errors_by_rank[rank])
            lines.append(
                f"  {label:>10} axis : mean={mean_e:5.2f}°  found={found_r*100:4.1f}%  (n={n})"
            )
        lines.append(f"  Visible frac       : {self.mean_fraction_visible*100:.1f}%")
        lines.append(f"  Score full/partial : {self.mean_score_full:.3f} / {self.mean_score_partial:.3f}")
        return "\n".join(lines)
 
 
# ═══════════════════════════════════════════════════════════════════════════════
#  1. Load ShapeNet mesh as point cloud
# ═══════════════════════════════════════════════════════════════════════════════
 
def load_shapenet_pcd(
    obj_path:  str,
    n_points:  int  = 4096,
    method:    str  = "poisson",   # "poisson" | "uniform"
    normalize: bool = True,
) -> o3d.geometry.PointCloud:
    """
    Load a ShapeNet model_normalized.obj and sample it into a point cloud.
 
    Parameters
    ----------
    obj_path  : path to model_normalized.obj
    n_points  : number of points to sample
    method    : "poisson" for Poisson-disk (more uniform) or "uniform" for random
    normalize : centre and scale to fit inside a unit sphere
 
    Returns
    -------
    o3d.geometry.PointCloud with normals
    """
    mesh = o3d.io.read_triangle_mesh(str(obj_path))
 
    if not mesh.has_vertices():
        raise ValueError(f"Could not load mesh or mesh is empty: {obj_path}")
 
    mesh.compute_vertex_normals()
 
    if method == "poisson":
        # init_factor controls the oversampling before thinning; 5 is a safe default
        pcd = mesh.sample_points_poisson_disk(
            number_of_points=n_points,
            init_factor=5,
        )
    else:
        pcd = mesh.sample_points_uniformly(number_of_points=n_points)
 
    if normalize:
        pts     = np.asarray(pcd.points)
        centroid = pts.mean(axis=0)
        scale    = np.linalg.norm(pts - centroid, axis=1).max()
        if scale > 1e-8:
            pcd.points = o3d.utility.Vector3dVector((pts - centroid) / scale)
 
    return pcd
 
 
# ═══════════════════════════════════════════════════════════════════════════════
#  2. Simulate partial view via hidden-point removal
# ═══════════════════════════════════════════════════════════════════════════════
 
def simulate_partial_view(
    pcd:            o3d.geometry.PointCloud,
    distance:       float = 2.5,
    radius_factor:  float = 100.0,
    seed:           Optional[int] = None,
) -> Tuple[o3d.geometry.PointCloud, np.ndarray]:
    """
    Pick a random viewpoint on a sphere of radius *distance* and remove all
    points not visible from that viewpoint using Open3D's hidden_point_removal.
 
    Parameters
    ----------
    pcd           : input point cloud (should be normalised to unit sphere)
    distance      : camera distance from origin (set > max cloud extent)
    radius_factor : HPR radius = distance * radius_factor; larger = more inclusive
    seed          : RNG seed for the random viewpoint
 
    Returns
    -------
    pcd_partial : visible-only point cloud
    camera_pos  : (3,) camera position used
    """
    rng   = np.random.default_rng(seed)
    # Uniform sphere sampling (Marsaglia / equal-area method)
    theta = rng.uniform(0.0, 2.0 * np.pi)
    phi   = np.arccos(rng.uniform(-1.0, 1.0))
    camera_pos = distance * np.array([
        np.sin(phi) * np.cos(theta),
        np.sin(phi) * np.sin(theta),
        np.cos(phi),
    ])
 
    hpr_radius = distance * radius_factor
    _, pt_map  = pcd.hidden_point_removal(camera_pos, hpr_radius)
 
    pcd_partial = pcd.select_by_index(pt_map)
    return pcd_partial, camera_pos
 
 
# ═══════════════════════════════════════════════════════════════════════════════
#  3. Compare multi-axis symmetry: each partial axis → nearest full axis
# ═══════════════════════════════════════════════════════════════════════════════
 
def match_symmetry_axes(
    multi_full:    MultiSymmetryResult,
    multi_partial: MultiSymmetryResult,
    max_match_deg: float = 45.0,
) -> List[AxisMatchResult]:
    """
    For each partial-cloud axis (ordered by score, best first), find the
    nearest full-cloud axis by angular distance.
 
    Multiple partial axes can match the same full axis — this is intentional.
    For example, if a chair has one strong symmetry plane and the partial view
    detects it twice (with slight noise), both should report a small error
    against the same full-cloud reference, not be penalised.
 
    Parameters
    ----------
    multi_full    : axes detected on full cloud (reference set)
    multi_partial : axes detected on partial cloud
    max_match_deg : error threshold; above this, found=False [degrees]
 
    Returns
    -------
    List[AxisMatchResult], one per partial axis, ordered by partial score rank
    """
    normals_f = multi_full.normals    # (K_f, 3)
    normals_p = multi_partial.normals # (K_p, 3)
    K_f = len(normals_f)
 
    matches: List[AxisMatchResult] = []
 
    for p_rank, (pn, p_ax) in enumerate(zip(normals_p, multi_partial.axes)):
        if K_f == 0:
            matches.append(AxisMatchResult(
                partial_rank=p_rank, partial_normal=pn, partial_score=p_ax.score,
                full_rank=0, full_normal=np.zeros(3), full_score=0.0,
                angular_error_deg=90.0, found=False,
            ))
            continue
 
        # Compute angle from this partial axis to every full axis
        angles = np.array([_plane_normal_angle_deg(pn, fn) for fn in normals_f])
        best_f = int(np.argmin(angles))
        err    = float(angles[best_f])
 
        matches.append(AxisMatchResult(
            partial_rank  = p_rank,
            partial_normal= pn,
            partial_score = p_ax.score,
            full_rank     = best_f,
            full_normal   = normals_f[best_f],
            full_score    = multi_full.axes[best_f].score,
            angular_error_deg = err,
            found         = err < max_match_deg,
        ))
 
    return matches
 
 
def compare_symmetry(
    pcd_full:      o3d.geometry.PointCloud,
    pcd_partial:   o3d.geometry.PointCloud,
    camera_pos:    np.ndarray,
    obj_path:      str   = "",
    k:             int   = 3,
    sym_kwargs:    Optional[dict] = None,
    max_match_deg: float = 45.0,
) -> ShapeSymmetryResult:
    """
    Detect K symmetry axes on both clouds, then match each partial axis to
    its nearest full-cloud axis.
 
    Parameters
    ----------
    pcd_full      : full point cloud (reference)
    pcd_partial   : partial (visible-only) point cloud
    camera_pos    : camera used for HPR (stored for reference)
    obj_path      : source path label
    k             : number of axes to detect per cloud
    sym_kwargs    : extra kwargs for detect_multi_symmetry_o3d()
    max_match_deg : angle above which a partial axis is considered not found
 
    Returns
    -------
    ShapeSymmetryResult
    """
    kw = dict(k=k, verbose=False)
    if sym_kwargs:
        kw.update(sym_kwargs)
 
    t0           = time.perf_counter()
    multi_full   = detect_multi_symmetry_o3d(pcd_full, **kw)
    elapsed_full = time.perf_counter() - t0
 
    t1              = time.perf_counter()
    multi_partial   = detect_multi_symmetry_o3d(pcd_partial, **kw)
    elapsed_partial = time.perf_counter() - t1
 
    matches = match_symmetry_axes(multi_full, multi_partial, max_match_deg=max_match_deg)
 
    errors        = [m.angular_error_deg for m in matches]
    found_frac    = float(np.mean([m.found for m in matches])) if matches else 0.0
 
    return ShapeSymmetryResult(
        obj_path          = obj_path,
        multi_full        = multi_full,
        n_points_full     = len(pcd_full.points),
        multi_partial     = multi_partial,
        n_points_partial  = len(pcd_partial.points),
        camera_pos        = camera_pos,
        axis_matches      = matches,
        errors_deg        = errors,
        found_fraction    = found_frac,
        mean_score_full   = float(np.mean(multi_full.scores))   if multi_full.k   else 0.0,
        mean_score_partial= float(np.mean(multi_partial.scores)) if multi_partial.k else 0.0,
        elapsed_full_s    = elapsed_full,
        elapsed_partial_s = elapsed_partial,
    )
 
 
# ═══════════════════════════════════════════════════════════════════════════════
#  4. Benchmark a single class folder
# ═══════════════════════════════════════════════════════════════════════════════
 
def benchmark_class(
    class_dir:      str,
    n_shapes:       Optional[int] = None,
    n_points:       int   = 4096,
    sample_method:  str   = "poisson",
    distance:       float = 2.5,
    radius_factor:  float = 100.0,
    k:              int   = 3,
    sym_kwargs:     Optional[dict] = None,
    max_match_deg:  float = 45.0,
    seed:           int   = 42,
    verbose:        bool  = True,
) -> ClassBenchmarkResult:
    """
    Run the full→partial multi-axis symmetry benchmark for one ShapeNet class folder.
 
    Parameters
    ----------
    class_dir     : path to the synset folder, e.g. '.../ShapeNet/03001627'
    n_shapes      : how many shapes to evaluate (None = all)
    n_points      : points to sample per mesh
    sample_method : "poisson" or "uniform"
    distance      : HPR camera distance
    radius_factor : HPR radius = distance * radius_factor
    k             : number of symmetry axes to detect per cloud
    sym_kwargs    : extra kwargs for detect_multi_symmetry_o3d()
    max_match_deg : angle threshold for declaring an axis pair matched
    seed          : base RNG seed
    verbose       : print per-shape progress
 
    Returns
    -------
    ClassBenchmarkResult
    """
    class_dir  = Path(class_dir)
    synset_id  = class_dir.name
    class_name = SHAPENET_SYNSETS.get(synset_id, synset_id)
 
    obj_files = sorted(glob.glob(
        str(class_dir / "**" / "models" / "model_normalized.obj"),
        recursive=True,
    ))
 
    if not obj_files:
        raise FileNotFoundError(
            f"No model_normalized.obj files found under {class_dir}\n"
            f"Expected: {class_dir}/<shape_id>/models/model_normalized.obj"
        )
 
    if n_shapes is not None:
        rng_sel   = np.random.default_rng(seed)
        obj_files = list(rng_sel.choice(obj_files, size=min(n_shapes, len(obj_files)), replace=False))
 
    if verbose:
        print(f"\n[benchmark] {class_name} ({synset_id}) | {len(obj_files)} shapes | k={k}")
 
    per_shape: List[ShapeSymmetryResult] = []
 
    for i, obj_path in enumerate(obj_files):
        shape_seed = seed + i
        try:
            pcd_full             = load_shapenet_pcd(obj_path, n_points=n_points, method=sample_method)
            pcd_partial, cam     = simulate_partial_view(pcd_full, distance=distance,
                                                         radius_factor=radius_factor, seed=shape_seed)
 
            if len(pcd_partial.points) < 50:
                if verbose:
                    print(f"  [{i+1}/{len(obj_files)}] SKIP (only {len(pcd_partial.points)} visible pts)")
                continue
 
            result = compare_symmetry(
                pcd_full, pcd_partial, cam,
                obj_path=obj_path, k=k,
                sym_kwargs=sym_kwargs,
                max_match_deg=max_match_deg,
            )
            per_shape.append(result)
 
            if verbose:
                frac = result.n_points_partial / result.n_points_full
                err_strs = "  ".join(
                    f"ax{m.partial_rank}:{m.angular_error_deg:4.1f}°{'✓' if m.found else '✗'}"
                    for m in result.axis_matches
                )
                print(
                    f"  [{i+1:>3}/{len(obj_files)}] vis={frac*100:.0f}%  {err_strs}"
                )
 
        except Exception as exc:
            if verbose:
                print(f"  [{i+1}/{len(obj_files)}] ERROR: {exc}")
 
    if not per_shape:
        raise RuntimeError(f"No shapes successfully processed in {class_dir}")
 
    # ── Per-rank aggregation ────────────────────────────────────────────────────
    # Collect errors per partial-axis rank across all shapes
    errors_by_rank: List[List[float]] = [[] for _ in range(k)]
    found_by_rank:  List[List[bool]]  = [[] for _ in range(k)]
 
    for r in per_shape:
        for m in r.axis_matches:
            if m.partial_rank < k:
                errors_by_rank[m.partial_rank].append(m.angular_error_deg)
                found_by_rank[m.partial_rank].append(m.found)
 
    all_errors = [e for rank_errors in errors_by_rank for e in rank_errors]
 
    result_cls = ClassBenchmarkResult(
        synset_id    = synset_id,
        class_name   = class_name,
        n_shapes     = len(per_shape),
        k            = k,
 
        errors_by_rank = errors_by_rank,
 
        mean_angular_error_deg = float(np.mean(all_errors)) if all_errors else 90.0,
        std_angular_error_deg  = float(np.std(all_errors))  if all_errors else 0.0,
 
        mean_error_by_rank = [
            float(np.mean(errs)) if errs else 90.0
            for errs in errors_by_rank
        ],
        found_rate_by_rank = [
            float(np.mean(founds)) if founds else 0.0
            for founds in found_by_rank
        ],
 
        mean_fraction_visible = float(np.mean([r.n_points_partial / r.n_points_full for r in per_shape])),
        mean_score_full       = float(np.mean([r.mean_score_full    for r in per_shape])),
        mean_score_partial    = float(np.mean([r.mean_score_partial for r in per_shape])),
 
        per_shape = per_shape,
    )
 
    if verbose:
        print(result_cls)
 
    return result_cls
 
 
# ═══════════════════════════════════════════════════════════════════════════════
#  5. Benchmark the full dataset (multiple classes)
# ═══════════════════════════════════════════════════════════════════════════════
 
def benchmark_dataset(
    shapenet_root: str,
    synset_ids:    Optional[List[str]] = None,
    n_per_class:   Optional[int] = 20,
    n_points:      int   = 4096,
    sample_method: str   = "poisson",
    distance:      float = 2.5,
    radius_factor: float = 100.0,
    k:             int   = 3,
    sym_kwargs:    Optional[dict] = None,
    max_match_deg: float = 45.0,
    seed:          int   = 42,
    verbose:       bool  = True,
) -> Dict[str, ClassBenchmarkResult]:
    """
    Run the benchmark across multiple ShapeNet classes and print a summary table.
 
    Parameters
    ----------
    shapenet_root : root of the ShapeNet dataset
    synset_ids    : list of synset IDs to benchmark (None = auto-discover all)
    n_per_class   : shapes per class (None = all)
    n_points      : points to sample per mesh
    sample_method : "poisson" or "uniform"
    distance      : HPR camera distance
    radius_factor : HPR radius factor
    k             : number of symmetry axes to detect per cloud
    sym_kwargs    : extra kwargs for detect_multi_symmetry_o3d()
    max_match_deg : angle threshold for axis matching
    seed          : base RNG seed
    verbose       : print per-class progress and summary table
 
    Returns
    -------
    dict mapping synset_id → ClassBenchmarkResult
    """
    shapenet_root = Path(shapenet_root)
 
    if synset_ids is None:
        synset_ids = [
            d.name for d in sorted(shapenet_root.iterdir())
            if d.is_dir() and not d.name.startswith(".")
        ]
 
    results: Dict[str, ClassBenchmarkResult] = {}
 
    for cls_seed_offset, sid in enumerate(synset_ids):
        class_dir = shapenet_root / sid
        if not class_dir.exists():
            if verbose:
                print(f"[benchmark] Skipping {sid}: folder not found.")
            continue
        try:
            r = benchmark_class(
                class_dir     = str(class_dir),
                n_shapes      = n_per_class,
                n_points      = n_points,
                sample_method = sample_method,
                distance      = distance,
                radius_factor = radius_factor,
                k             = k,
                sym_kwargs    = sym_kwargs,
                max_match_deg = max_match_deg,
                seed          = seed + cls_seed_offset * 10000,
                verbose       = verbose,
            )
            results[sid] = r
        except Exception as exc:
            if verbose:
                print(f"[benchmark] ERROR on {sid}: {exc}")
 
    if verbose and results:
        _print_summary_table(results)
 
    return results
 
 
def _print_summary_table(results: Dict[str, ClassBenchmarkResult]) -> None:
    """Print a per-rank summary table across all classes."""
    # Determine max k across all results
    max_k = max(r.k for r in results.values())
 
    rank_labels = {0: "primary", 1: "secondary", 2: "tertiary"}
 
    # Header
    rank_cols = "  ".join(
        f"{'err_' + rank_labels.get(i, f'r{i}'):>12}  {'found%':>6}"
        for i in range(max_k)
    )
    header = f"\n{'Class':<16} {'Synset':<12} {'N':>4}  {'OverallErr°':>11}  {rank_cols}  {'Vis%':>5}"
    print(header)
    print("─" * len(header))
 
    for sid, r in results.items():
        rank_vals = "  ".join(
            (f"{r.mean_error_by_rank[i]:>12.2f}°  {r.found_rate_by_rank[i]*100:>5.1f}%"
             if i < len(r.mean_error_by_rank) else f"{'N/A':>13}  {'N/A':>6}")
            for i in range(max_k)
        )
        print(
            f"{r.class_name:<16} {sid:<12} {r.n_shapes:>4}  "
            f"{r.mean_angular_error_deg:>10.2f}°  {rank_vals}  {r.mean_fraction_visible*100:>5.1f}%"
        )
 
    # Overall row
    all_errors = np.concatenate([
        [e for rank_errors in cls.errors_by_rank for e in rank_errors]
        for cls in results.values()
    ])
    n_total = sum(r.n_shapes for r in results.values())
    print("─" * len(header))
    print(f"{'OVERALL':<16} {'':<12} {n_total:>4}  {all_errors.mean():>10.2f}°")
    print()
 
 

 
# ═══════════════════════════════════════════════════════════════════════════════
#  Convenience: process a single OBJ end-to-end
# ═══════════════════════════════════════════════════════════════════════════════
 
def evaluate_single(
    obj_path:      str,
    n_points:      int   = 4096,
    distance:      float = 2.5,
    radius_factor: float = 100.0,
    k:             int   = 3,
    seed:          int   = 0,
    sym_kwargs:    Optional[dict] = None,
    max_match_deg: float = 45.0,
    verbose:       bool  = True,
) -> ShapeSymmetryResult:
    """
    Load one ShapeNet OBJ, simulate a partial view, and compare K symmetry axes.
 
    Example
    -------
        result = evaluate_single(
            '/home/jvermandere/datasets/ShapeNet/03001627/'
            '1a6f615e8b1b5ae4dbbc9440457e303e/models/model_normalized.obj',
            k=3,
        )
        print(result)
    """
    pcd_full = load_shapenet_pcd(obj_path, n_points=n_points)
    pcd_partial, cam = simulate_partial_view(
        pcd_full, distance=distance, radius_factor=radius_factor, seed=seed,
    )
 
    if verbose:
        frac = len(pcd_partial.points) / len(pcd_full.points)
        print(f"Loaded  : {len(pcd_full.points)} pts from {Path(obj_path).parents[1].name}")
        print(f"Visible : {len(pcd_partial.points)} pts ({frac*100:.1f}%) from camera {cam.round(3)}")
 
    result = compare_symmetry(
        pcd_full, pcd_partial, cam,
        obj_path=obj_path, k=k,
        sym_kwargs=sym_kwargs, max_match_deg=max_match_deg,
    )
 
    if verbose:
        print(result)
 
    return result, pcd_partial

In [ ]:
# Single shape
result = evaluate_single('/home/jvermandere/datasets/ShapeNet/03001627/f595abef9bc7320944b2fa2cac0778f5/models/model_normalized.obj')


In [ ]:
pcd_partial,cam = simulate_partial_view(load_shapenet_pcd('/home/jvermandere/datasets/ShapeNet/03001627/f595abef9bc7320944b2fa2cac0778f5/models/model_normalized.obj'))
drm.visualise_open3d(pcd_partial).show()

In [ ]:
sym = detect_symmetry_o3d(pcd_partial, threshold=0.02, n_random_candidates=600)

In [ ]:
# Get Open3D geometries for your visualiser
geoms = symmetry_geometries(sym, pcd_partial)          # axis + plane + reflected cloud
scene = drm.visualise_open3d([pcd_partial] + geoms)       # your existing helper
scene.show()

In [ ]:

# One class
r = benchmark_class('/home/jvermandere/datasets/ShapeNet/03001627', n_shapes=5, n_points=4096)


In [ ]:

# Full dataset (multiple classes)
results = benchmark_dataset(
    shapenet_root='/path/to/ShapeNet',
    synset_ids=['03001627', '04379243', '02691156'],  # chair, table, airplane
    n_per_class=20,
)